In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# 1. HR documents
documents = [
    "Employees receive 26 annual leave days.",
    "Leave requests must be approved by the manager.",
    "Employees have supplementary health insurance.",
    "Monthly salaries are paid at the end of each month.",
    "Employees can work remotely two days per week.",
    "Overtime payments are calculated according to company policy.",
    "Employees can request unpaid leave.",
    "Health insurance coverage includes medical services."
]

# 2. Load embedding model
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# 3. Generate document embeddings
document_embeddings = model.encode(
    documents,
    convert_to_numpy=True,
    normalize_embeddings=True
)

# 4. Retrieval function
def retrieve_chunks(query: str, top_k: int = 3, score_threshold: float = 0.0):

    # Generate query embedding
    query_embedding = model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Calculate cosine similarity
    scores = cosine_similarity(
        query_embedding,
        document_embeddings
    )[0]

    # Rank documents
    ranked_indices = np.argsort(scores)[::-1]

    # Retrieve relevant chunks
    results = []

    for index in ranked_indices:
        score = float(scores[index])

        if score >= score_threshold:
            results.append({
                "text": documents[index],
                "score": score
            })

        if len(results) == top_k:
            break

    return results


# 5. Test retrieval
queries = [
    "How many annual leave days do employees receive?",
    "What health insurance benefits are available?",
    "When are employee salaries paid?"
]

for query in queries:

    print(f"\nQuery: {query}")
    print("-" * 60)

    results = retrieve_chunks(
        query=query,
        top_k=3,
        score_threshold=0.0
    )

    for rank, result in enumerate(results, start=1):
        print(f"{rank}. {result['text']}")
        print(f"   Similarity: {result['score']:.4f}")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1024.93it/s]
`SentenceTransformer._target_device` has been deprecated. Please use `SentenceTransformer.device` instead.
`SentenceTransformer._target_device` has been deprecated. Please use `SentenceTransformer.device` instead.



Query: How many annual leave days do employees receive?
------------------------------------------------------------
